# Create Wolfson Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

The Wolfson Foundation is a major UK grant-making charity (founded 1955) funding research and capital projects across **Science & Medicine**, **Health & Disability**, **Education**, and **Arts & Humanities**. This ingest covers the foundation's **published grant record, 2014-2025**, one row per grant.

**Method 1 (direct open-data download).** The foundation is a [360Giving](https://www.threesixtygiving.org/) publisher (registry prefix `360G-wolfson`). The scraper (`scripts/local/wolfson_to_s3.py`) downloads the full grant corpus directly from the foundation's own site as an Excel workbook published under **CC BY 4.0** (`wolfson.org.uk/.../2014-2025-grants-awarded.xlsx`), in the standard 360Giving schema. No browser automation, no rate-limited search API — a single authoritative file.

**Awarding body:** Wolfson Foundation - F4320320670 (GB, ROR 0333xzh65, DOI 10.13039/501100001320)

**Source:** the 360Giving `grants` sheet. This is an **org-level grant funder** — each grant is made to a recipient **organization** (no named principal investigator). Each grant carries a stable `Identifier`, a `Title`, a `Description`, an `Amount Awarded` (GBP), a real `Award Date`, the `Recipient Org: Name` / `City` / `Country`, and a `Grant Programme: Title`.

**Schema choices:**
- One row per grant. `funder_award_id` = the 360Giving `Identifier`, e.g. `360G-wolfson-19832` (stable, unique, source-authoritative).
- `display_name` = the grant's own `Title` (source-authoritative, 100% present).
- `funding_type` = `grant` uniformly.
- `funder_scheme` = the `Grant Programme` (Arts and Humanities, Education, Health and Disability, Science and Medicine, ...).
- **Amount** = 360Giving `Amount Awarded` in GBP → `amount` (DOUBLE) + `currency` = `GBP` where published (> 0); NULL otherwise. **§6.7 is NOT waived** — populated wherever the foundation posts a positive figure (~55% of grants), never imputed; any `0`/blank is treated as NULL (the foundation publishes `0` where the amount is undisclosed).
- `start_date` = the real `Award Date` (a true `YYYY-MM-DD` date — no false precision); `start_year` derived from it.
- `end_date` / `end_year` = NULL. The source publishes only a *planned* duration (on ~32% of grants), not an actual grant end date, so no false-precision end is asserted.
- `lead_investigator` = an **org-only** lead: `given_name`/`family_name` NULL and `affiliation.name` = the recipient organization. `affiliation.country` = the source's `Recipient Org: Country` mapped to ISO (e.g. `UK` → `GB`) — **source-authoritative**, not guessed.
- `co_lead_investigator` and `investigators` are NULL (the source names no individuals).
- `description` = the grant's `Description`.

**Coverage note:** the corpus is the foundation's complete published record for **2014-2025** (4,254 grants). Programmes span all Wolfson funding (Arts & Humanities, Education, Health & Disability, Science & Medicine), so not every grant is research — but the funder is in OpenAlex (3,091 works) and the research grants carry the same authoritative `Identifier` for downstream matching.

**S3 location:** `s3a://openalex-ingest/awards/wolfson/wolfson_grants.parquet`

**Provenance:** `wolfson_foundation` (verified count=0 on production 2026-06-01).


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wolfson_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wolfson/wolfson_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.wolfson_raw;


## Step 1.5: Inspect raw + programme/year/coverage

Expected: 4,254 grants, 2014-2025. title / funder_award_id / award_date / start_year / recipient_org / recipient_country_iso / grant_programme / description 100%; amount ~55.3% (the foundation publishes 0 where undisclosed; §6.7 not waived).


In [ ]:
%sql
DESCRIBE openalex.awards.wolfson_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.wolfson_raw LIMIT 5;


In [ ]:
%sql
-- Per-(programme, start_year) row counts + amount coverage.
SELECT grant_programme, start_year, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       COUNT(recipient_org) AS has_org
FROM openalex.awards.wolfson_raw
GROUP BY grant_programme, start_year
ORDER BY start_year DESC, rows DESC;


In [ ]:
%sql
-- Top recipient organizations (sanity check parsing didn't bleed fields).
SELECT recipient_org, recipient_country_iso, COUNT(*) AS rows
FROM openalex.awards.wolfson_raw
WHERE recipient_org IS NOT NULL
GROUP BY recipient_org, recipient_country_iso
ORDER BY rows DESC
LIMIT 15;


## Step 1.6: Fail-fast - verify Wolfson Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320670;


## Step 2: Transform to award schema

`display_name` = the grant's own `Title`. `start_date` = the real `Award Date`. `lead_investigator` is an org-only lead: `given_name`/`family_name` NULL, `affiliation.name` = recipient org, `affiliation.country` = the source's recipient country mapped to ISO (source-authoritative). `co_lead_investigator` / `investigators` are NULL.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wolfson_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320670  -- Wolfson Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    COALESCE(n.title, CONCAT('Wolfson Foundation grant ', n.funder_award_id)) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN 'GBP' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.grant_programme AS funder_scheme,
    'wolfson_foundation' AS provenance,
    TRY_CAST(n.award_date AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.recipient_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.recipient_org AS name,
                n.recipient_country_iso AS country,  -- source-authoritative (Recipient Org: Country, UK->GB); never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CAST(NULL AS STRING) AS landing_page_url,  -- 360Giving has no per-grant page; programme URL kept in raw only
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.wolfson_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 155


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wolfson_foundation' AND priority = 155;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    155 AS priority  -- Wolfson Foundation grants
FROM openalex.awards.wolfson_awards;


## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived, partial): the foundation publishes a positive amount on ~55% of grants and `0` where undisclosed (the `$0`-guard nulls those), so `pct_amount` ≈ 55%. Other completeness checks: display_name / description / start_year / lead (recipient org) ~100%; start_date populated from the real Award Date; currency = [GBP]; lead_investigator.affiliation.country = GB (source-authoritative).


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.wolfson_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.wolfson_awards;


In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.wolfson_awards;


In [ ]:
%sql
-- §6.7 amount check (NOT waived, partial). Expect currency = [GBP].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.wolfson_awards;


In [ ]:
%sql
-- Programme (scheme) split.
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(amount) AS has_amount,
       ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.wolfson_awards
GROUP BY funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Country sanity: lead_investigator.affiliation.country (source-authoritative).
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.wolfson_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY rows DESC;


In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.wolfson_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       funder_scheme AS programme, funding_type, start_date, start_year, amount, currency,
       lead_investigator.affiliation.name AS org,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.wolfson_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;
